In [ ]:
# prepare_training.ipynb
# Stage 1 of 2: build + embed the review H5. Run this FIRST; then run
# training.ipynb (same pod). Splitting keeps data-prep and training separate so
# you can re-run training without rebuilding data.
REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG = "/workspace/stable_query_latent_logs/pipeline.log"
print('repo:', REPO)
print('log :', LOG)

In [ ]:
# Clone or pull before every run.
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git remote -v
!git pull origin main
!git status --short
!git rev-parse HEAD


In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor.
import subprocess, threading, time, psutil
from pathlib import Path

stop = False

def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == "max":
            return None
        return int(text)
    except Exception:
        return None

def get_memory_status():
    limit = _read_int("/sys/fs/cgroup/memory.max")
    used = _read_int("/sys/fs/cgroup/memory.current")
    if limit is None or used is None:
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, "cgroup"
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, "host"

def get_gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        parts = [p.strip() for p in out.splitlines()[0].split(",")]
        return f"{float(parts[0]):.0f}%, {float(parts[1])/1024:.1f}/{float(parts[2])/1024:.1f} GiB"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] {ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()


In [ ]:
# Build text H5.
!python -u game_review_data/build.py   --data-dir game_review_data/build_new_gamedata   --only metadata split text-h5   --split-device cuda   --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Embed text H5.
!python -u game_review_data/embedding_incloud.py   --input-h5 game_review_data/build_new_gamedata/text_h5.h5   --output-h5 game_review_data/embedding_h5.h5   --device cuda   --text-load-chunk-size 500000   --tok-workers 4   --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Data prepared. Verify the embedding H5 before moving to training.ipynb.
from pathlib import Path
h5 = Path(REPO, 'game_review_data/embedding_h5.h5')
size_gb = (h5.stat().st_size / 1e9) if h5.exists() else 0.0
print(f'embedding_h5: {h5}\n  exists={h5.exists()}  size={size_gb:.2f} GB')
assert h5.exists(), 'embedding_h5.h5 missing -- re-run the build + embed cells'
print('Data ready -> run training.ipynb on this pod.')